# Hidden Markov Model for Biological Sequence Prediction

This notebook demonstrates the use of a Hidden Markov Model (HMM) and the Viterbi algorithm for identifying hidden biological regions from a DNA sequence.

The model predicts the most likely hidden states based on observed nucleotides.

In [1]:
import numpy as np
import pandas as pd
import math

## Defining Hidden States

The hidden states represent biological regions present in a gene sequence.

States included in this model:

B  -> Start state

E  -> Exon

J  -> Junction site

I  -> Intron

T  -> Termination state

In [2]:
states = {
    "B": 0,
    "E": 1,
    "J": 2,
    "I": 3,
    "T": 4
}

state_names = {
    0: "B",
    1: "E",
    2: "J",
    3: "I",
    4: "T"
}

print(states)

{'B': 0, 'E': 1, 'J': 2, 'I': 3, 'T': 4}


## Transition Probability Matrix

The transition matrix defines the probability of moving from one hidden state to another.

For example:

- Exons may continue for several positions
- Junction sites often transition into introns

In [3]:
transition_probs = np.array([

    [0.0, 1.0, 0.0, 0.0, 0.0],

    [0.0, 0.90, 0.10, 0.0, 0.0],

    [0.0, 0.0, 0.0, 1.0, 0.0],

    [0.0, 0.0, 0.0, 0.93, 0.07],

    [0.0, 0.0, 0.0, 0.0, 0.0]

])

transition_df = pd.DataFrame(
    transition_probs,
    index=states.keys(),
    columns=states.keys()
)

transition_df

,B,E,J,I,T
B,0.0,1.0,0.0,0.00,0.00
E,0.0,0.9,0.1,0.00,0.00
J,0.0,0.0,0.0,1.00,0.00
I,0.0,0.0,0.0,0.93,0.07
T,0.0,0.0,0.0,0.00,0.00


## Emission Probability Matrix

Emission probabilities define the likelihood of observing nucleotides from a particular hidden state.

Observed symbols:
A, C, G, and T

In [4]:
symbols = {
    "A": 0,
    "C": 1,
    "G": 2,
    "T": 3
}

emission_probs = np.array([

    [0.00, 0.00, 0.00, 0.00],

    [0.28, 0.22, 0.32, 0.18],

    [0.01, 0.00, 0.99, 0.00],

    [0.42, 0.08, 0.12, 0.38],

    [0.00, 0.00, 0.00, 0.00]

])

emission_df = pd.DataFrame(
    emission_probs,
    index=states.keys(),
    columns=["A", "C", "G", "T"]
)

emission_df

,A,C,G,T
B,0.00,0.00,0.00,0.00
E,0.28,0.22,0.32,0.18
J,0.01,0.00,0.99,0.00
I,0.42,0.08,0.12,0.38
T,0.00,0.00,0.00,0.00


## Input DNA Sequence

The following DNA sequence is provided as the observed sequence for analysis.

In [5]:
sequence = "ATGCGTACGTTAGCGTACCTGA"

print(sequence)

ATGCGTACGTTAGCGTACCTGA


## Initializing Viterbi Matrices

Two matrices are created:

1. Probability matrix
2. Backtracking matrix

The probability matrix stores the highest probability values.

The backtracking matrix stores the previous best state for reconstruction of the optimal path.

In [6]:
num_states = len(states)

seq_length = len(sequence)

viterbi = np.full(
    (num_states, seq_length),
    -np.inf
)

traceback = np.zeros(
    (num_states, seq_length),
    dtype=int
)

print(viterbi)

[[-inf -inf -inf -inf -inf -inf -inf -inf -inf -inf -inf -inf -inf -inf
  -inf -inf -inf -inf -inf -inf -inf -inf]
 [-inf -inf -inf -inf -inf -inf -inf -inf -inf -inf -inf -inf -inf -inf
  -inf -inf -inf -inf -inf -inf -inf -inf]
 [-inf -inf -inf -inf -inf -inf -inf -inf -inf -inf -inf -inf -inf -inf
  -inf -inf -inf -inf -inf -inf -inf -inf]
 [-inf -inf -inf -inf -inf -inf -inf -inf -inf -inf -inf -inf -inf -inf
  -inf -inf -inf -inf -inf -inf -inf -inf]
 [-inf -inf -inf -inf -inf -inf -inf -inf -inf -inf -inf -inf -inf -inf
  -inf -inf -inf -inf -inf -inf -inf -inf]]


## Initial Step

The first nucleotide is initialized using exon emission probabilities.

In [7]:
first_char = sequence[0]

viterbi[1][0] = (
    math.log(1.0)
    +
    math.log(
        emission_probs[1][
            symbols[first_char]
        ]
    )
)

print(viterbi)

[[       -inf        -inf        -inf        -inf        -inf        -inf
         -inf        -inf        -inf        -inf        -inf        -inf
         -inf        -inf        -inf        -inf        -inf        -inf
         -inf        -inf        -inf        -inf]
 [-1.27296568        -inf        -inf        -inf        -inf        -inf
         -inf        -inf        -inf        -inf        -inf        -inf
         -inf        -inf        -inf        -inf        -inf        -inf
         -inf        -inf        -inf        -inf]
 [       -inf        -inf        -inf        -inf        -inf        -inf
         -inf        -inf        -inf        -inf        -inf        -inf
         -inf        -inf        -inf        -inf        -inf        -inf
         -inf        -inf        -inf        -inf]
 [       -inf        -inf        -inf        -inf        -inf        -inf
         -inf        -inf        -inf        -inf        -inf        -inf
         -inf        -inf        

## Viterbi Scoring Function

This function calculates the highest probability for each state at every sequence position.

In [8]:
def compute_state_score(current_state, current_position):

    max_score = -np.inf
    best_state = 0

    observed_char = sequence[current_position]

    for previous_state in range(num_states):

        transition_value = transition_probs[
            previous_state
        ][current_state]

        emission_value = emission_probs[
            current_state
        ][
            symbols[observed_char]
        ]

        if transition_value == 0 or emission_value == 0:
            continue

        score = (
            viterbi[previous_state][current_position - 1]
            +
            math.log(transition_value)
            +
            math.log(emission_value)
        )

        if score > max_score:

            max_score = score
            best_state = previous_state

    return max_score, best_state

## Filling the Viterbi Matrix

Each position in the sequence is evaluated using transition and emission probabilities.

In [9]:
for pos in range(1, seq_length):

    for state in range(num_states):

        best_score, previous = compute_state_score(
            state,
            pos
        )

        viterbi[state][pos] = best_score

        traceback[state][pos] = previous

## Displaying Viterbi Probability Matrix

In [10]:
viterbi_df = pd.DataFrame(
    viterbi,
    index=states.keys()
)

viterbi_df

,0,1,2,3,4,5,6,7,8,9,...,12,13,14,15,16,17,18,19,20,21
B,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,...,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf
E,-1.272966,-3.093125,-4.337919,-5.957408,-7.202202,-9.022361,-10.400688,-12.020176,-13.264971,-15.085130,...,-19.528410,-21.147898,-22.392693,-24.212852,-25.591178,-27.210666,-28.830154,-30.650313,-31.895108,-33.273434
J,-inf,-inf,-5.405760,-inf,-8.270043,-inf,-15.930117,-inf,-14.332811,-inf,...,-20.596250,-inf,-23.460533,-inf,-31.120607,-inf,-inf,-inf,-32.962949,-38.802863
I,-inf,-inf,-inf,-7.931489,-10.124323,-9.237627,-10.177698,-12.775998,-14.968832,-15.300395,...,-19.473456,-22.071755,-24.264589,-24.428117,-25.368188,-27.966488,-30.564787,-31.604942,-33.797776,-33.830449
T,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,...,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf,-inf


## Displaying Traceback Matrix

In [11]:
traceback_df = pd.DataFrame(
    traceback,
    index=states.keys()
)

traceback_df

,0,1,2,3,4,5,6,7,8,9,...,12,13,14,15,16,17,18,19,20,21
B,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
E,0,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
J,0,0,1,0,1,0,1,0,1,0,...,1,0,1,0,1,0,0,0,1,1
I,0,0,0,2,3,2,3,3,3,2,...,3,3,3,2,3,3,3,3,3,2
T,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## Backtracking to Find Optimal Path

The traceback process identifies the most probable hidden state sequence.

In [12]:
def decode_path():

    last_position = seq_length - 1

    current_state = np.argmax(
        viterbi[:, last_position]
    )

    path = [current_state]

    for pos in range(last_position, 0, -1):

        current_state = traceback[
            current_state
        ][pos]

        path.append(current_state)

    path.reverse()

    decoded_states = [
        state_names[s]
        for s in path
    ]

    return decoded_states

## Final Predicted Hidden State Sequence

In [13]:
predicted_path = decode_path()

print("Observed Sequence:")
print(sequence)

print("\nPredicted Hidden States:")
print("".join(predicted_path))

Observed Sequence:
ATGCGTACGTTAGCGTACCTGA

Predicted Hidden States:
EEEEEEEEEEEEEEEEEEEEEE


# Conclusion

The Viterbi algorithm successfully predicts the most probable hidden biological regions from the observed DNA sequence.

This approach is commonly used in:

- Gene prediction
- Sequence analysis
- Bioinformatics research
- Computational biology